In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

In [0]:
df_visit = spark.read.table('healthcare.silver.visit')

In [0]:
df_patient = spark.read.table('healthcare.silver.patient').drop('_rescued_data','updated_at')
df_hospital = spark.read.table('healthcare.silver.hospital').drop('_rescued_data','updated_at')
df_diagnosis = spark.read.table('healthcare.silver.diagnosis').drop('_rescued_data','updated_at')

In [0]:
df_visit_details = (df_visit
                    .join(df_patient,'patient_id','Left')
                    .join(df_hospital,'hospital_id','Left')
                    .join(df_diagnosis,'diagnosis_code','Left')
                    )

In [0]:
df = (df_visit_details.withColumn('last_admission_date',lag('admission_date').over(Window.partitionBy('patient_id').orderBy('admission_date')))
      .withColumn('Admission_Gap_Days',datediff('admission_date','last_admission_date')    
      )
      .withColumn('Re-admission_flag', when(col('Admission_Gap_Days')<30,1).otherwise(0))
      )

In [0]:
df_gold =( 
        df.groupBy(
            'hospital_id',
            'hospital_name',
            'diagnosis_desc'         
               )
            .agg(
            count('*').alias('Total_visits'),
            sum('Re-admission_flag').alias('total_readmission'),
            round(sum('Re-admission_flag')/count('*'),2).alias('avg_cost_per_visit'),
            sum('cost').alias('total_cost'),
            avg('cost').alias('avg_cost')
            )
        )


In [0]:
df_gold.write.format('delta').mode('overwrite').saveAsTable('healthcare.gold.gold_kpi_data')